# 기계 고장 예측 분류 모델 (예지보전)
**학번_이름** | 모델: **분류** | 데이터: 예시 데이터 〈기계 고장〉 (10,000행 × 14열)

- **예측할 것**: 공정 센서 값(온도, 회전 속도, 토크, 공구 마모 시간)을 보고 그 기계가 **고장(1)인지 정상(0)인지** 분류
- **왜?**: 산업공학의 설비보전(TPM)·예지보전. 고장이 나기 전에 미리 알면 갑작스러운 라인 정지와 수리 비용을 줄일 수 있음
- **기계학습인 이유**: 고장 여부를 바로 계산하는 공식이 없음. 여러 센서 값이 함께 어떻게 변할 때 고장이 나는지를 데이터에서 배워야 함

## 1. 라이브러리 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

## 2. 데이터 불러오기
수업 화면에서 받은 〈기계 고장〉 CSV 파일을 코랩 왼쪽 📁 폴더에 올리고, 아래 `DATA_PATH`를 그 파일 이름으로 바꿔요.

In [ ]:
DATA_PATH = '학번_이름_데이터.csv'   # ← 내 파일 이름으로 바꾸기

df = pd.read_csv(DATA_PATH)
print('크기:', df.shape)
df.head()

## 3. 데이터 살펴보기

In [ ]:
df.info()
print('\n빈 값 개수:\n', df.isnull().sum())

In [ ]:
print(df['Machine failure'].value_counts())
print('고장 비율: %.1f%%' % (df['Machine failure'].mean()*100))
df['Machine failure'].value_counts().plot(kind='bar', color=['steelblue','tomato'])
plt.title('Machine failure (0=normal, 1=failure)'); plt.show()

➡️ 고장이 전체의 약 3%밖에 안 돼요(**불균형 데이터**). 모델이 무조건 '정상'이라고만 해도 정확도가 97%가 나오므로, **정확도만 보면 안 되고 재현율(실제 고장을 얼마나 잡았나)** 을 꼭 봐야 해요.

In [ ]:
num_cols = ['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, c in zip(axes, num_cols):
    df.boxplot(column=c, by='Machine failure', ax=ax)
    ax.set_title(c); ax.set_xlabel('failure')
plt.suptitle(''); plt.tight_layout(); plt.show()

➡️ 고장 난 기계는 토크가 크거나, 회전 속도가 낮거나, 공구 마모 시간이 긴 경우가 많아요. 하나의 값만으로 딱 잘리지 않아서 모델이 여러 값을 함께 봐야 해요.

## 4. 전처리
1. **필요 없는 열 빼기**: `UDI`(번호), `Product ID`(제품 번호)는 고장과 관계없는 이름표예요.
2. **정답을 알려 주는 열 빼기**: `TWF, HDF, PWF, OSF, RNF`는 '어떤 종류의 고장이었는지'예요. 고장이 난 뒤에야 알 수 있는 값이라 넣으면 모델이 답을 미리 보고 푸는 셈(**데이터 누수**)이 돼요.
3. **글자를 숫자로**: `Type`(제품 품질 등급 L/M/H)을 0/1/2로 바꿔요.
4. **새 특징 만들기**(산업공학 지식 활용):
   - 온도차 = 공정 온도 − 공기 온도 (열이 잘 안 빠지면 과열 고장)
   - 동력(W) = 토크 × 각속도 (너무 작거나 크면 전력 고장)
   - 마모×토크 = 공구 마모 × 토크 (공구에 걸리는 부담, 과부하 고장)

In [ ]:
data = df.drop(columns=['UDI','Product ID','TWF','HDF','PWF','OSF','RNF'])
data = data.dropna().drop_duplicates()
data['Type'] = data['Type'].map({'L':0, 'M':1, 'H':2})

data['Temp diff [K]'] = data['Process temperature [K]'] - data['Air temperature [K]']
data['Power [W]']     = data['Torque [Nm]'] * data['Rotational speed [rpm]'] * 2*np.pi/60
data['Wear x Torque'] = data['Tool wear [min]'] * data['Torque [Nm]']

X = data.drop(columns=['Machine failure'])
y = data['Machine failure']
print('입력 특징:', list(X.columns))
X.head()

## 5. 훈련용 / 평가용 나누기
80%로 학습하고, 처음 보는 20%로 평가해요. `stratify=y`로 두 묶음의 고장 비율을 똑같이 맞춰요(고장이 적어서 한쪽에 몰리지 않게).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('훈련:', X_train.shape, ' 평가:', X_test.shape)
print('평가용 고장 수:', y_test.sum())

In [ ]:
def evaluate(name, y_true, y_pred):
    r = {'모델': name,
         '정확도': accuracy_score(y_true, y_pred),
         '정밀도': precision_score(y_true, y_pred, zero_division=0),
         '재현율': recall_score(y_true, y_pred),
         'F1': f1_score(y_true, y_pred)}
    print(f"[{name}] 정확도 {r['정확도']:.3f} | 정밀도 {r['정밀도']:.3f} | 재현율 {r['재현율']:.3f} | F1 {r['F1']:.3f}")
    return r
results = []

## 6. 비교 기준: 무조건 '정상'이라고 말하는 모델

In [ ]:
results.append(evaluate('기준(모두 정상)', y_test, np.zeros(len(y_test), dtype=int)))

➡️ 정확도는 높지만 재현율이 0이에요. 고장을 하나도 못 잡으니 쓸모가 없어요.

## 7. 모델 1: 결정 트리
'토크가 60보다 큰가?' 같은 예/아니오 질문을 이어 가며 정상/고장을 나눠요.

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
results.append(evaluate('결정 트리', y_test, dt.predict(X_test)))

## 8. 모델 2: 랜덤 포레스트 (최종 모델)
결정 트리를 200개 만들어 **다수결**로 정해요. 트리마다 데이터와 특징을 조금씩 다르게 뽑아서 배우므로 한 개보다 안정적이에요.
`class_weight='balanced'`: 수가 적은 '고장'을 틀리면 더 크게 벌점을 줘서 고장을 놓치지 않게 해요.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                            class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
results.append(evaluate('랜덤 포레스트', y_test, rf.predict(X_test)))

## 9. 성능 개선: 판단 기준(임곗값) 조정
랜덤 포레스트는 '고장일 확률'을 내놓아요. 기본은 50%가 넘으면 고장이라고 하지만,
공장에서는 **고장을 놓치는 것(설비 정지, 큰 수리비)** 이 **괜히 점검하는 것(잠깐의 점검 비용)** 보다 훨씬 손해이므로 기준을 낮춰 고장을 더 많이 잡아요.

In [ ]:
proba = rf.predict_proba(X_test)[:, 1]
print('임곗값 | 정밀도 | 재현율 |  F1')
for t in [0.5, 0.4, 0.3, 0.2]:
    p = (proba >= t).astype(int)
    print(f'  {t:.1f}  | {precision_score(y_test,p,zero_division=0):.3f}  | {recall_score(y_test,p):.3f}  | {f1_score(y_test,p):.3f}')

In [ ]:
THRESHOLD = 0.3   # 위 표를 보고 재현율과 정밀도가 균형 잡힌 값으로 고르기
y_final = (proba >= THRESHOLD).astype(int)
results.append(evaluate(f'랜덤 포레스트(임곗값 {THRESHOLD})', y_test, y_final))

## 10. 결과 비교와 혼동 행렬

In [ ]:
pd.DataFrame(results).set_index('모델').round(3)

In [ ]:
cm = confusion_matrix(y_test, y_final)
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=16,
                 color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.xticks([0,1], ['pred normal','pred failure']); plt.yticks([0,1], ['real normal','real failure'])
plt.title('Confusion matrix'); plt.colorbar(); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f'실제 고장 {tp+fn}대 중 {tp}대를 미리 잡음, {fn}대는 놓침')
print(f'정상인데 고장이라고 해서 괜히 점검: {fp}대')
print(classification_report(y_test, y_final, target_names=['정상','고장']))

## 11. 어떤 특징이 중요했나 (특징 중요도)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot(kind='barh', color='seagreen', figsize=(7,5))
plt.title('Feature importance (Random Forest)'); plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))

## 12. 새 기계 상태로 예측해 보기

In [ ]:
new = pd.DataFrame([
    # Type, 공기온도, 공정온도, 회전속도, 토크, 공구마모
    [0, 298.1, 308.6, 1551, 42.8, 30],    # 평범한 상태
    [0, 302.5, 311.0, 1300, 65.0, 210],   # 토크 크고 공구가 많이 닳음
], columns=['Type','Air temperature [K]','Process temperature [K]',
            'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]'])
new['Temp diff [K]'] = new['Process temperature [K]'] - new['Air temperature [K]']
new['Power [W]']     = new['Torque [Nm]'] * new['Rotational speed [rpm]'] * 2*np.pi/60
new['Wear x Torque'] = new['Tool wear [min]'] * new['Torque [Nm]']
p = rf.predict_proba(new[X.columns])[:, 1]
for i, v in enumerate(p):
    print(f'기계 {i+1}: 고장 확률 {v*100:.1f}% → {"⚠️ 점검 필요" if v >= THRESHOLD else "✅ 정상 가동"}')

## 13. 정리
- 무조건 '정상'이라고 하면 정확도는 높지만 고장을 하나도 못 잡아요. 그래서 **재현율과 F1**을 중심으로 평가했어요.
- 랜덤 포레스트 + `class_weight='balanced'` + 임곗값 조정으로 실제 고장을 대부분 미리 잡았어요.
- 중요한 특징은 마모×토크, 동력, 회전 속도, 토크 등이었고, 이는 공구 과부하·전력 이상·과열 같은 실제 고장 원리와 맞아요.
- 활용: 공장 설비의 센서 값을 실시간으로 넣어 고장 확률이 기준을 넘으면 점검 알림 → **계획 보전**으로 갑작스러운 라인 정지 감소